# MSBA 502: Understanding Hotel Booking Cancellations
**Team:** Sofia Song and Alicia Osuna Fernandez

**Business question:** Which bookings are most likely to cancel, and what can reservation and revenue managers do about it?

**Data:** Hotel Booking Demand (Antonio, Almeida & Nunes, 2019), 119,390 bookings, city + resort hotel, 2015-2017.

### Code contribution (edit honestly)
| Section | Who | % |
|---|---|---|
| 1. Load data, first cancellation summaries (your original cells) | [name] | [ ]% |
| 2. Data quality and cleaning | [name] | [ ]% |
| 3. Exploratory analysis and charts | [name] | [ ]% |
| 4. Logistic regression and evaluation | [name] | [ ]% |
| 5. Time-based check and risk tiers | [name] | [ ]% |
| **Total per person** | Sofia: [ ]% / Alicia: [ ]% | 100% |

## 1. Load data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# In Colab: upload the CSV. Elsewhere: place hotel_bookings.csv next to this notebook.
try:
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]   # Colab renames repeats, e.g. "hotel_bookings (1).csv"
except ImportError:
    filename = "hotel_bookings.csv"

raw = pd.read_csv(filename)
print("Dataset size:", raw.shape)
print("Overall cancellation rate (raw):", round(raw["is_canceled"].mean() * 100, 2), "%")

## 2. Data quality and cleaning
We check missing values, duplicate rows, and impossible values, then document every decision.

In [ ]:
# --- 2a. Missing values ---
missing = raw.isna().sum()
print("Columns with missing values:")
print(missing[missing > 0])

# --- 2b. Duplicates ---
n_dups = raw.duplicated().sum()
print(f"\nExact duplicate rows: {n_dups:,} ({n_dups / len(raw) * 100:.1f}% of the data)")

# --- 2c. Impossible / extreme values (flagged on the raw data) ---
guests = raw["adults"] + raw["children"].fillna(0) + raw["babies"]
print("Bookings with zero guests:", (guests == 0).sum())
print("Bookings with negative adr:", (raw["adr"] < 0).sum())
print("Bookings with adr > 1000:", (raw["adr"] > 1000).sum())
print("Bookings with zero nights:",
      ((raw["stays_in_week_nights"] + raw["stays_in_weekend_nights"]) == 0).sum())

In [ ]:
df = raw.copy()

# Fill small gaps with defensible values
df["children"] = df["children"].fillna(0)           # only 4 missing; assume no children
df["country"] = df["country"].fillna("Unknown")
df = df.drop(columns=["agent", "company"])           # many missing / ID codes, not used

# Remove duplicates, then impossible values
df = df.drop_duplicates()
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["total_stay"] = df["stays_in_week_nights"] + df["stays_in_weekend_nights"]

df = df[df["total_guests"] > 0]                      # a booking needs at least one guest
df = df[df["adr"].between(0, 1000)]                  # drops negative and extreme rates
df = df[df["total_stay"] > 0]                        # zero-night stays are not real stays

# Calendar helpers
month_order = ["January", "February", "March", "April", "May", "June", "July",
               "August", "September", "October", "November", "December"]
df["arrival_date_month"] = pd.Categorical(df["arrival_date_month"],
                                          categories=month_order, ordered=True)

print(f"Rows before: {len(raw):,}  ->  after cleaning: {len(df):,}")
print("Cancellation rate after cleaning:", round(df["is_canceled"].mean() * 100, 2), "%")

In [ ]:
# Sensitivity: how much does the duplicate decision matter?
# (Same cleaning as above, but WITHOUT dropping duplicates)
alt = raw.copy()
alt["children"] = alt["children"].fillna(0)
alt["total_guests"] = alt["adults"] + alt["children"] + alt["babies"]
alt["total_stay"] = alt["stays_in_week_nights"] + alt["stays_in_weekend_nights"]
alt = alt[(alt["total_guests"] > 0) & alt["adr"].between(0, 1000) & (alt["total_stay"] > 0)]

compare = pd.DataFrame({
    "with_duplicates_%": alt.groupby("market_segment")["is_canceled"].mean() * 100,
    "deduplicated_%": df.groupby("market_segment")["is_canceled"].mean() * 100,
}).round(1)
compare.loc["ALL"] = [alt["is_canceled"].mean() * 100, df["is_canceled"].mean() * 100]
print(compare.round(1))

## 3. Exploratory analysis
Cancellation rates by hotel, lead time, market segment, season, price, and stay length.

In [ ]:
# Overall and by hotel (includes the significance test from the original notebook,
# now computed from the data instead of typed-in numbers)
hotel_tab = df.groupby("hotel")["is_canceled"].agg(["sum", "count", "mean"])
hotel_tab["rate_%"] = (hotel_tab["mean"] * 100).round(2)
print(hotel_tab[["count", "rate_%"]])

x = hotel_tab["sum"].values
n = hotel_tab["count"].values
p_pool = x.sum() / n.sum()
z = (x[0] / n[0] - x[1] / n[1]) / np.sqrt(p_pool * (1 - p_pool) * (1 / n[0] + 1 / n[1]))
p_value = 2 * (1 - stats.norm.cdf(abs(z)))
print(f"\nTwo-proportion z-test (city vs resort): z = {z:.2f}, p = {p_value:.4g}")

ax = (hotel_tab["mean"] * 100).plot(kind="bar", color="teal", rot=0)
ax.set_title("Cancellation Rate by Hotel Type")
ax.set_ylabel("Cancellation Rate (%)")
ax.set_ylim(0, 60)
for i, v in enumerate(hotel_tab["mean"] * 100):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Lead time
df["lead_time_group"] = pd.cut(
    df["lead_time"], bins=[-1, 7, 30, 90, 180, 365, np.inf],
    labels=["0-7", "8-30", "31-90", "91-180", "181-365", "366+"])
lead_rate = df.groupby("lead_time_group", observed=False)["is_canceled"].mean() * 100

ax = lead_rate.plot(kind="bar", color="orange", rot=0)
ax.set_title("Cancellation Rate by Lead Time (days)")
ax.set_ylabel("Cancellation Rate (%)")
plt.tight_layout()
plt.show()
print(lead_rate.round(1))

In [ ]:
# Market segment, with a chi-square test of independence
seg = df[df["market_segment"] != "Undefined"]
seg_rate = (seg.groupby("market_segment")["is_canceled"].mean() * 100).sort_values()

ax = seg_rate.plot(kind="barh", color="steelblue")
ax.set_title("Cancellation Rate by Market Segment")
ax.set_xlabel("Cancellation Rate (%)")
plt.tight_layout()
plt.show()

chi2, p, dof, _ = stats.chi2_contingency(pd.crosstab(seg["market_segment"], seg["is_canceled"]))
print(f"Chi-square test (segment vs cancellation): chi2 = {chi2:.1f}, dof = {dof}, p = {p:.4g}")

In [ ]:
# Seasonality: cancellation rate and booking volume by arrival month, by hotel
month = (df.groupby(["arrival_date_month", "hotel"], observed=False)["is_canceled"]
           .mean().unstack() * 100)
ax = month.plot(marker="o", figsize=(9, 4))
ax.set_title("Cancellation Rate by Arrival Month")
ax.set_ylabel("Cancellation Rate (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Price (adr) and length of stay
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

df.boxplot(column="adr", by="is_canceled", ax=axes[0], showfliers=False)
axes[0].set_title("Average Daily Rate")
axes[0].set_xlabel("0 = kept, 1 = canceled")
axes[0].set_ylabel("adr")

stay = df.assign(stay_group=pd.cut(df["total_stay"], [0, 1, 2, 3, 5, 7, np.inf],
                                   labels=["1", "2", "3", "4-5", "6-7", "8+"]))
(stay.groupby("stay_group", observed=False)["is_canceled"].mean() * 100).plot(
    kind="bar", ax=axes[1], color="slategray", rot=0)
axes[1].set_title("Cancellation Rate by Length of Stay (nights)")
axes[1].set_ylabel("Cancellation Rate (%)")

fig.suptitle("")
plt.tight_layout()
plt.show()

print(df.groupby("is_canceled")[["adr", "total_stay", "lead_time"]].median())

In [ ]:
# Deposit type: a data caution, not a model input (see Section 4)
dep = df.groupby("deposit_type")["is_canceled"].agg(["count", "mean"])
dep["mean"] = (dep["mean"] * 100).round(1)
print(dep.rename(columns={"mean": "cancellation_rate_%"}))

## 4. Logistic regression model
**Prediction point:** at the time of booking. We only use information a manager would know then.

**Excluded on purpose**
- `reservation_status`, `reservation_status_date`: directly reveal the outcome (leakage).
- `deposit_type`: "Non Refund" bookings show an implausible ~99% cancellation rate, which looks like a recording quirk in this dataset, not real guest behavior.
- `assigned_room_type`, `booking_changes`, `days_in_waiting_list`, `required_car_parking_spaces`, `total_of_special_requests`: can change or be recorded after booking.
- `country`, `reserved_room_type`: many categories; kept out to keep the model simple and explainable.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, roc_auc_score, roc_curve, classification_report)

numeric_features = ["lead_time", "adr", "total_stay", "total_guests",
                    "previous_cancellations", "previous_bookings_not_canceled"]
categorical_features = ["hotel", "market_segment", "distribution_channel",
                        "customer_type", "arrival_date_month", "is_repeated_guest"]

work = df[df["market_segment"] != "Undefined"].copy()
work["arrival_date_month"] = work["arrival_date_month"].astype(str)
work["is_repeated_guest"] = work["is_repeated_guest"].astype(str)

X = work[numeric_features + categorical_features]
y = work["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

def make_model():
    prep = ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features),
    ])
    return Pipeline([("prep", prep), ("logit", LogisticRegression(max_iter=1000))])

model = make_model().fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

baseline = 1 - y_test.mean()
print(f"Baseline (always predict 'not canceled'): accuracy = {baseline:.3f}")
print(f"Accuracy : {accuracy_score(y_test, pred):.3f}")
print(f"Precision: {precision_score(y_test, pred):.3f}")
print(f"Recall   : {recall_score(y_test, pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, proba):.3f}")

cm = pd.DataFrame(confusion_matrix(y_test, pred),
                  index=["Actual: kept", "Actual: canceled"],
                  columns=["Predicted: kept", "Predicted: canceled"])
print("\nConfusion matrix (threshold 0.5):")
print(cm)

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"Logistic regression (AUC = {roc_auc_score(y_test, proba):.2f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random guessing")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC Curve (test set)")
plt.legend()
plt.tight_layout()
plt.show()

# Precision/recall trade-off at different cutoffs (helps pick the "flag" threshold)
rows = []
for t in [0.2, 0.25, 0.3, 0.4, 0.5]:
    p_t = (proba >= t).astype(int)
    rows.append({"threshold": t,
                 "flagged_%": round(p_t.mean() * 100, 1),
                 "precision": round(precision_score(y_test, p_t), 3),
                 "recall": round(recall_score(y_test, p_t), 3)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# What drives cancellation? Odds ratios.
# Numeric variables are standardized, so their odds ratio is "per 1 standard deviation".
# Categorical odds ratios compare each category with the reference category (dropped one).
names = model.named_steps["prep"].get_feature_names_out()
coefs = model.named_steps["logit"].coef_[0]
odds = (pd.DataFrame({"feature": names, "odds_ratio": np.exp(coefs)})
          .assign(feature=lambda d: d["feature"].str.replace("num__", "").str.replace("cat__", ""))
          .sort_values("odds_ratio"))

print("Factors that LOWER cancellation odds the most:")
print(odds.head(8).round(2).to_string(index=False))
print("\nFactors that RAISE cancellation odds the most:")
print(odds.tail(8).iloc[::-1].round(2).to_string(index=False))

top = pd.concat([odds.head(6), odds.tail(6)])
plt.figure(figsize=(7, 5))
plt.barh(top["feature"], top["odds_ratio"] - 1, left=1,
         color=["seagreen" if v < 1 else "indianred" for v in top["odds_ratio"]])
plt.axvline(1, color="black", lw=1)
plt.xlabel("Odds ratio (1 = no effect)")
plt.title("Strongest Cancellation Predictors")
plt.tight_layout()
plt.show()

## 5. Robustness check and risk tiers
**Time-based split:** train on earlier arrivals (2015-2016), test on the most recent year (2017). This is closer to how a hotel would actually use the model.

In [ ]:
train_mask = work["arrival_date_year"] <= 2016
test_mask = work["arrival_date_year"] == 2017

model_t = make_model().fit(X[train_mask], y[train_mask])
proba_t = model_t.predict_proba(X[test_mask])[:, 1]
pred_t = (proba_t >= 0.5).astype(int)

print(f"Train rows: {train_mask.sum():,}   Test rows: {test_mask.sum():,}")
print(f"Accuracy : {accuracy_score(y[test_mask], pred_t):.3f}")
print(f"Precision: {precision_score(y[test_mask], pred_t):.3f}")
print(f"Recall   : {recall_score(y[test_mask], pred_t):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y[test_mask], proba_t):.3f}")

In [ ]:
# Risk tiers for managers: group predicted probability into Low / Medium / High.
# Check that the tiers line up with real cancellation rates on bookings the model has not seen.
tiers = pd.DataFrame({"proba": proba, "actual": y_test.values})
tiers["risk_tier"] = pd.cut(tiers["proba"], bins=[0, 0.2, 0.4, 1.0],
                            labels=["Low (<20%)", "Medium (20-40%)", "High (>40%)"],
                            include_lowest=True)
summary = tiers.groupby("risk_tier", observed=False).agg(
    bookings=("actual", "size"),
    actual_cancellation_rate=("actual", "mean"))
summary["share_of_bookings_%"] = (summary["bookings"] / summary["bookings"].sum() * 100).round(1)
summary["actual_cancellation_rate"] = (summary["actual_cancellation_rate"] * 100).round(1)
print(summary)

ax = summary["actual_cancellation_rate"].plot(kind="bar", color=["seagreen", "goldenrod", "indianred"], rot=0)
ax.set_title("Actual Cancellation Rate by Predicted Risk Tier (test set)")
ax.set_ylabel("Actual cancellation rate (%)")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 6. Key takeaways
- **Duplicates matter.** About 27% of raw rows are exact duplicates. Removing them (plus invalid bookings) drops the overall cancellation rate from 37.0% to about 27.7%, and shrinks the Groups segment's rate from 61% to about 27%. Reporting the raw numbers would overstate the problem.
- **Hotel type:** City hotel about 30% vs resort about 24% (significant, z-test).
- **Lead time:** cancellation climbs from about 8.5% (0-7 days) to about 41% (366+ days); canceled bookings have a median lead time of 80 days vs 38 for kept bookings.
- **Segment:** Online travel agencies are the highest-risk segment (about 36%); Corporate, Direct and Offline TA/TO bookings are around 12-15%.
- **Seasonality:** cancellations peak in summer (about 32% in July-August) and are lowest in November-January (about 21-23%).
- **Trend:** the cancellation rate rises each arrival year (about 21% in 2015, 27% in 2016, 32% in 2017), so a model trained on older data will understate recent risk.
- **Model:** ROC-AUC about 0.73 on a random split and about 0.72 on the time-based split. At the default 0.5 cutoff, accuracy (74%) barely beats the always-predict-"kept" baseline (72%) and recall is only about 19%. Lowering the cutoff to about 0.3 catches roughly two-thirds of cancellations while flagging about 40% of bookings. Choose the cutoff based on the cost of a follow-up vs a missed cancellation.
- **Risk tiers (cutoffs 20% / 40%):** the High tier is about 20% of bookings and cancels about 52% of the time; the Low tier is about 36% of bookings and cancels about 12% of the time. That is a roughly 4-5x difference, which is the basis for targeting follow-up.

**Limitations:** two hotels, 2015-2017; associations, not causes; duplicate handling and the `deposit_type` exclusion are judgment calls; the model has moderate, not strong, predictive power.
**Future (MSBA 503):** newer multi-hotel data, tree-based models, and a revenue-at-risk estimate (risk x room rate x nights).